# Agent的高级用法

## 1、设置agent的名称

使用字段name来进行设置

举例：

In [1]:
from langchain.agents import create_agent
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
import os
load_dotenv(override=True)
from rich import print as rprint

# 以init_chat_model为例
# model = init_chat_model(
#     model="gpt-5.4-mini",
#     model_provider="openai",
#     api_key=os.getenv("CLOSEAI_API_KEY"),
#     base_url=os.getenv("CLOSEAI_BASE_URL")
# )

model = init_chat_model(
    model="deepseek-v4-flash",
    model_provider="deepseek",
    api_key=os.getenv("DEEPSEEK_API_KEY"),
    base_url=os.getenv("DEEPSEEK_BASE_URL")
)

agent = create_agent(
    model=model,
    name="agent01"
)

# 调用
response= agent.invoke({
    "messages":[
        {"role":"system","content":"你是一个精通数学的老师，擅长以通俗易懂的方式讲解数学问题"},
        {"role":"user","content":"100 + 20 * 3 = ？"}
    ]
})


rprint(response)

for message in response["messages"]:
    message.pretty_print()

{
    'messages': [
        SystemMessage(
            content='你是一个精通数学的老师，擅长以通俗易懂的方式讲解数学问题',
            additional_kwargs={},
            response_metadata={},
            id='3524fcb1-10e0-4365-9f06-aed0fb0295e1'
        ),
        HumanMessage(
            content='100 + 20 * 3 = ？',
            additional_kwargs={},
            response_metadata={},
            id='2c54d217-6ca4-482c-96e9-d665dc003f07'
        ),
        AIMessage(
            content='100 + 20 × 3  \n先算乘法：20 × 3 = 60  \n再算加法：100 + 60 = 160  \n\n所以答案是：**160**',
            additional_kwargs={
                'refusal': None,
                'reasoning_content': '我们需要回答中文。用户问 100 + 20 * 3 = ？ 
需要注意运算顺序，先乘后加：20*3=60，100+60=160。需要以通俗易懂方式。可以简短说明。'
            },
            response_metadata={
                'token_usage': {
                    'completion_tokens': 88,
                    'prompt_tokens': 54,
                    'total_tokens': 142,
                    'completion_tokens_details': {
                        'accepted_prediction_tokens': None,
                        'audio_tokens': None,
                        'reasoning_tokens': 49,
                        'rejected_prediction_tokens': None
                    },
                    'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0},
                    'prompt_cache_hit_tokens': 0,
                    'prompt_cache_miss_tokens': 54
                },
                'model_provider': 'deepseek',
                'model_name': 'deepseek-flash',
                'system_fingerprint': 'aeb56401ca74e127821c4f9126dcb669',
                'id': '00c0ba7e-02ec-4a35-a92e-71dc63a3e6b4',
                'finish_reason': 'stop',
                'logprobs': None
            },
            name='agent01',
            id='lc_run--01a12dbe-ab9b-7910-8ebe-5c93019d0b04-0',
            tool_calls=[],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 54,
                'output_tokens': 88,
                'total_tokens': 142,
                'input_token_details': {'cache_read': 0},
                'output_token_details': {'reasoning': 49}
            }
        )
    ]
}

================================ System Message ================================

你是一个精通数学的老师，擅长以通俗易懂的方式讲解数学问题
================================ Human Message =================================

100 + 20 * 3 = ？
================================== Ai Message ==================================
Name: agent01

100 + 20 × 3  
先算乘法：20 × 3 = 60  
再算加法：100 + 60 = 160  

所以答案是：**160**


## 2、系统提示词的设置

使用 system_prompt 参数进行设置，可以是 str, 也可以是 SystemMessage

输出的 messages 里面并不会有 SystemMessage。

举例1：并行调用

In [3]:
from langchain_tavily import TavilySearch
from langchain.agents import create_agent
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
import os
load_dotenv(override=True)
from rich import print as rprint

# 1.导入模型
# model = init_chat_model(
#     model="gpt-5.4-mini",
#     model_provider="openai",
#     api_key=os.getenv("CLOSEAI_API_KEY"),
#     base_url=os.getenv("CLOSEAI_BASE_URL")
# )

model = init_chat_model(
    model="deepseek-v4-flash",
    model_provider="deepseek",
    api_key=os.getenv("DEEPSEEK_API_KEY"),
    base_url=os.getenv("DEEPSEEK_BASE_URL")
)

# 2.导入工具
web_search = TavilySearch(max_results=2)
## TavilySearch 不传 tavily_api_key 也能运行，是因为它内置了自动从环境变量读取密钥的机制。代码中的 load_dotenv(override=True) 将 .env 文件里的 TAVILY_API_KEY 加载到了系统环境中，TavilySearch 在初始化时会自动获取这个值，所以不需要手动传参。

# 3.创建Agent
agent = create_agent(
    model=model,
    tools=[web_search],
    system_prompt="你是一名多才多艺的智能助手，可以调用工具帮助用户解决问题。"
)

# 4.运行Agent获得结果
result = agent.invoke(
    {"messages": [
        {"role": "user", "content": "请帮我查询2026年足球世界杯是哪个国家举办的？"}
    ]}
)

rprint(result)

{
    'messages': [
        HumanMessage(
            content='请帮我查询2026年足球世界杯是哪个国家举办的？',
            additional_kwargs={},
            response_metadata={},
            id='b5524579-cced-44fe-8b33-7682910a4159'
        ),
        AIMessage(
            content='我来帮您查询2026年足球世界杯的举办国家。',
            additional_kwargs={
                'refusal': None,
                'reasoning_content': 'The user is asking about which country hosts the 2026 FIFA World Cup. Let me 
search for this information.'
            },
            response_metadata={
                'token_usage': {
                    'completion_tokens': 118,
                    'prompt_tokens': 1855,
                    'total_tokens': 1973,
                    'completion_tokens_details': {
                        'accepted_prediction_tokens': None,
                        'audio_tokens': None,
                        'reasoning_tokens': 23,
                        'rejected_prediction_tokens': None
                    },
                    'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 1664},
                    'prompt_cache_hit_tokens': 1664,
                    'prompt_cache_miss_tokens': 191
                },
                'model_provider': 'deepseek',
                'model_name': 'deepseek-flash',
                'system_fingerprint': 'aeb56401ca74e127821c4f9126dcb669',
                'id': '1f883654-0cbb-4718-9641-f91786e378d7',
                'finish_reason': 'tool_calls',
                'logprobs': None
            },
            id='lc_run--01a12dc1-fc5d-7162-ae7e-fd743af5e138-0',
            tool_calls=[
                {
                    'name': 'tavily_search',
                    'args': {'query': '2026年足球世界杯 举办国家'},
                    'id': 'call_00_q5tx6IhKtcXW4dwKCpu31324',
                    'type': 'tool_call'
                },
                {
                    'name': 'tavily_search',
                    'args': {'query': '2026 FIFA World Cup host country'},
                    'id': 'call_01_yVejUcHXpug5kweHbmqg0376',
                    'type': 'tool_call'
                }
            ],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 1855,
                'output_tokens': 118,
                'total_tokens': 1973,
                'input_token_details': {'cache_read': 1664},
                'output_token_details': {'reasoning': 23}
            }
        ),
        ToolMessage(
            content='{"query": "2026年足球世界杯 举办国家", "follow_up_questions": null, "answer": null, "images": 
[], "results": [{"url": 
"https://zh.wikipedia.org/zh-hans/2026%E5%B9%B4%E5%9C%8B%E9%9A%9B%E8%B6%B3%E5%8D%94%E4%B8%96%E7%95%8C%E7%9B%83", 
"title": "2026年国际足协世界杯 - 维基百科，自由的百科全书", "content": "2026年国际足协世界杯（英语：2026 FIFA World
Cup Canada/Mexico/United States，西班牙语：Copa mundial de la FIFA Canadá/México/Estados Unidos 2026，法语：Coupe 
du monde de la FIFA de 2026 
Canada/Mexique/États-Unis），汉语圈简称“美加墨世界杯”，为第23届国际足协世界杯赛事，定于2026年6月11日至7月19日举行，
赛事在加拿大、墨西哥、美国共16座城市展开，为世界杯首次由3国联合举办。\\n\\n本届世界杯是继2002年赛事后再次由多国联合
主办，第4次由中北美洲及加勒比海地区主办。墨西哥此前曾举办1970年和1986年世界杯，本届赛事后成为首个3办世界杯之国；美
国曾于1994年举办赛事；加拿大则是首次举办男子世界杯，恰巧的是三者均在本届世界杯淘汰赛十六强出局。此外，本届赛事将回
归传统的北半球夏季档期。 [...] |  FIFA World Cup 26  Copa Mundial de la FIFA 2026  Coupe du Monde FIFA 2026 |\\n|  
We Are 26  Somos 26  Nous Sommes 26 |\\n| 赛事资料 |\\n| 主办国家／地区 | 加拿大 墨西哥 美国 |\\n| 比赛日期 | 
6月11日–7月19日 |\\n| 参赛队数 | 48 队（来自6个大洲） |\\n| 球场 | 16 个（位于16个城市） |\\n| 最终成绩 |\\n| 冠军 
| 西班牙（第 2 次夺冠） |\\n| 亚军 | 阿根廷 |\\n| 季军 | 英格兰 |\\n| 殿军 | 法国 |\\n| 赛事统计 |\\n| 比赛场数 | 
104 场 |\\n| 总入球数 | 308 球（场均 2.96 球） |\\n| 入场人数 | 6,810,966 人（场均 65,490 人） |\\n| 最佳射手 | 
法国 基利安·麦巴比（10 球） |\\n| 最佳球员 | 西班牙 罗德里 |\\n| 最佳守门员 \\"守门员 (足球)\\") | 西班牙 乌奈·西蒙
|\\n| 最佳年轻球员 | 西班牙 保·库巴西 | [...] 2026年国际足总世界杯申办投票\\n\\n| 国家 | 投票 |\\n| 第1轮 |\\n| 
加拿大／ 墨西哥／ 美国 | 134 |\\n| 摩洛哥 | 65 |\\n| 白票 | 1 |\\n| 弃权 | 3 |\\n| 总票数（不包括弃权） | 200 
|\\n\\n## 
比赛场地\\n\\n[编辑]\\n\\n2022年6月16日，国际足联公布了2026年世界杯的16个举办城市，其中加拿大2个，墨西哥3个，美国11
个。尽管赛事由三

**为什么会出现连续两条 ToolMessage?**

- 模型在一次回复里发起了多个 tool_calls（并行工具调用）。

    - 一条 AIMessage 里可以包含 多个 tool_calls（OpenAI / DeepSeek 等都支持 parallel tool calls）。

    - 这里 AIMessage.tool_calls 里就一次性带了两个 tool_call。

        ```python
        tool_calls = [
            {  # 第一次
                'name': 'tavily_search',
                'args': {'query': '2024年诺贝尔物理学奖得主', ...},
                'id': 'call_00_gV6x3g3XCWDxEviUq7OS4935',
            },
            {  # 第二次
                'name': 'tavily_search',
                'args': {'query': '2024 Nobel Prize in Physics winners announcement', ...},
                'id': 'call_01_8KBqInWgbIjuemaWjNex9015',
            },
        ]
        ```
    - 说明大模型决定进行两次工具调用，一次用中文搜索，一次用英文搜索。

- 有几个 tool_call，Agent 就会执行几次工具，并紧接着追加几条 ToolMessage。

- 这些 ToolMessage 是“连续”的，因为它们都对应同一条 AIMessage 里的不同 tool_call_id。


**并行调用**

只进行一轮工具调用为例，可能并行调用工具，可能是同一个工具并行调用，也可能是不同的工具并行调用，判断是不是“并行调用”，**不看工具是不是同一个，而看这些 tool_call 是不是出现在同一条 `AIMessage` 里**。


**“并行”指的是模型侧，不代表运行时一定真并发**

- 模型侧：一次生成多个 tool_calls，这是并行工具调用。

- 执行侧：LangChain / LangGraph 的 ToolNode 可能用线程池或 asyncio.gather 并发执行，也可能顺序执行，取决于框架实现和工具类型（同步/异步）。

但无论执行是并发还是顺序，语义上都属于“同一批工具调用”，所以仍叫并行工具调用。

模型侧大模型可能会选择并行调用，也可能选择循环调用，这取决于具体情况。

举例2：循环调用

In [4]:
from langchain.agents import create_agent
from langchain_core.messages import SystemMessage
from langchain_core.tools import tool
from rich import print as rprint

# 工具：实现两数相加
@tool
def add_numbers(a: int, b: int) -> str:
    """计算并返回两个数的和。"""
    return f"和为：{a + b}"


# 创建客服助手Agent
agent = create_agent(
    model=model,
    tools=[add_numbers],  # 工具列表
    # system_prompt="你是一个数学助手，解决日常的算术问题"
    system_prompt=SystemMessage(content="你是一个数学助手，解决日常的算术问题")
)

response = agent.invoke(
    {"messages": [
        {"role": "user", "content": "10加上20再加上30是多少？"}
    ]},
)

rprint(response)
# print(response["messages"][-1].content)

{
    'messages': [
        HumanMessage(
            content='10加上20再加上30是多少？',
            additional_kwargs={},
            response_metadata={},
            id='71381d4d-8fef-41dc-aab0-302cc3bd6ac0'
        ),
        AIMessage(
            content='我来帮您计算。',
            additional_kwargs={
                'refusal': None,
                'reasoning_content': 'The user asks: 10 + 20 + 30. Let me use the tool.\n\nFirst 10 + 20 = 30, then
30 + 30 = 60. Two calls dependent though. Let me call add_numbers(10,20) first.\n\nActually I can chain: first call
add(10,20), then add(result,30). These are dependent, so sequential.'
            },
            response_metadata={
                'token_usage': {
                    'completion_tokens': 140,
                    'prompt_tokens': 319,
                    'total_tokens': 459,
                    'completion_tokens_details': {
                        'accepted_prediction_tokens': None,
                        'audio_tokens': None,
                        'reasoning_tokens': 82,
                        'rejected_prediction_tokens': None
                    },
                    'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0},
                    'prompt_cache_hit_tokens': 0,
                    'prompt_cache_miss_tokens': 319
                },
                'model_provider': 'deepseek',
                'model_name': 'deepseek-flash',
                'system_fingerprint': 'aeb56401ca74e127821c4f9126dcb669',
                'id': 'df026cd4-3a59-4903-b42c-8393dff9034c',
                'finish_reason': 'tool_calls',
                'logprobs': None
            },
            id='lc_run--01a12dce-8583-7182-9448-89fa8a3b3b8e-0',
            tool_calls=[
                {
                    'name': 'add_numbers',
                    'args': {'a': 10, 'b': 20},
                    'id': 'call_00_QRMVLX3TnswBiKbhNRee2106',
                    'type': 'tool_call'
                }
            ],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 319,
                'output_tokens': 140,
                'total_tokens': 459,
                'input_token_details': {'cache_read': 0},
                'output_token_details': {'reasoning': 82}
            }
        ),
        ToolMessage(
            content='和为：30',
            name='add_numbers',
            id='943b450d-b34e-49de-9473-b6e856ad63dc',
            tool_call_id='call_00_QRMVLX3TnswBiKbhNRee2106'
        ),
        AIMessage(
            content='',
            additional_kwargs={'refusal': None, 'reasoning_content': 'Now add 30 + 30.'},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 62,
                    'prompt_tokens': 474,
                    'total_tokens': 536,
                    'completion_tokens_details': {
                        'accepted_prediction_tokens': None,
                        'audio_tokens': None,
                        'reasoning_tokens': 8,
                        'rejected_prediction_tokens': None
                    },
                    'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 256},
                    'prompt_cache_hit_tokens': 256,
                    'prompt_cache_miss_tokens': 218
                },
                'model_provider': 'deepseek',
                'model_name': 'deepseek-flash',
                'system_fingerprint': 'aeb56401ca74e127821c4f9126dcb669',
                'id': 'aafa1846-054e-4cba-8d3e-527632711935',
                'finish_reason': 'tool_calls',
                'logprobs': None
            },
            id='lc_run--01a12dce-8a75-71e2-afb1-6bd9ed5a275e-0',
            tool_calls=[
                {
                    'name': 'add_numbers',
                    'args': {'a': 30, 'b': 30},
                    'id': 'call_00_FoxvJOMgSAJ19Yr9j6lI2641',
                    'type': 'tool_call'
                }
         

举例3：重试机制（也属循环调用）

In [5]:
from langchain.agents import create_agent
from langchain.tools import tool
from langchain.messages import SystemMessage, HumanMessage

flag = 0

@tool
def get_weather(city: str):
    """
    天气查询工具

    Args:
        city: 城市名称
    """
    global flag
    flag += 1

    if flag < 3:
        # raise Exception("暂时无法访问")
        return "TEMP_UNAVAILABLE: 天气服务暂时不可用，请稍后重试"

    return f"{city}今天天气挺好"


messages = [
    HumanMessage("你好，杭州今天的天气如何？")
]

agent = create_agent(
    model=model,
    tools=[get_weather],
    system_prompt=SystemMessage(
        "你是一个天气助手。"
        "当工具返回以 'TEMP_UNAVAILABLE:' 开头的结果时，"
        "说明是临时故障，不要立即放弃；"
        "你应再次调用同一个工具，最多重试 3 次。"
        "如果 3 次后仍失败，再向用户说明服务暂时不可用。"
    )
)
response = agent.invoke({"messages": messages})
# print(response)
for msg in response["messages"]:
    msg.pretty_print()

================================ Human Message =================================

你好，杭州今天的天气如何？
================================== Ai Message ==================================
Tool Calls:
  get_weather (call_00_zhb74xdCOIg94Rnx8rNJ8736)
 Call ID: call_00_zhb74xdCOIg94Rnx8rNJ8736
  Args:
    city: 杭州
================================= Tool Message =================================
Name: get_weather

TEMP_UNAVAILABLE: 天气服务暂时不可用，请稍后重试
================================== Ai Message ==================================
Tool Calls:
  get_weather (call_00_URUZoR3KsNK4EXB6PhON5776)
 Call ID: call_00_URUZoR3KsNK4EXB6PhON5776
  Args:
    city: 杭州
================================= Tool Message =================================
Name: get_weather

TEMP_UNAVAILABLE: 天气服务暂时不可用，请稍后重试
================================== Ai Message ==================================
Tool Calls:
  get_weather (call_00_U29puhkAMMcbrLAf1BII0104)
 Call ID: call_00_U29puhkAMMcbrLAf1BII0104
  Args:
    city: 杭州
==================

## 3、结构化输出的4种策略

### 3.1 ProviderStrategy策略：

In [10]:
from pydantic import BaseModel, Field
from langchain.agents import create_agent
from langchain.agents.structured_output import ProviderStrategy
from langchain.messages import HumanMessage
from rich import print as rprint
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
import os

load_dotenv(override=True)

# 1.模型初始化
# model = init_chat_model(
#     model="gpt-5.4-mini",
#     model_provider="openai",
#     api_key=os.getenv("CLOSEAI_API_KEY"),
#     base_url=os.getenv("CLOSEAI_BASE_URL")
# )

model = init_chat_model(
    model="deepseek-v4-flash",
    model_provider="deepseek",
    # reasoning_effort="none",  # 关闭思考模式
    extra_body={"thinking": {"type": "disabled"}},  # 第二种写法关闭思考模式
    api_key=os.getenv("DEEPSEEK_API_KEY"),
    base_url=os.getenv("DEEPSEEK_BASE_URL")
)

# 2.使用 Pydantic 结构化方式定义
class ContractInfo(BaseModel):
    """用户的联系方式"""
    name : str = Field(description="用户的姓名")
    email : str = Field(description="用户的邮箱")
    phone : str = Field(description="用户的电话")

agent = create_agent(
    model = model,
    response_format=ProviderStrategy(ContractInfo)
)


response = agent.invoke({
    "messages": [
        {"role":"user", "content":"从以下信息中提取用户信息，小明的邮箱是shkstart@atguigu.com,电话是13012341234"}
    ]
})

rprint(response)

BadRequestError: Error code: 400 - {'error': {'message': 'This response_format type is unavailable now (request_id: d8c959ee-71bb-4601-a9bc-dbbc1271788a)', 'type': 'invalid_request_error', 'param': None, 'code': 'invalid_request_error'}}

你遇到的 BadRequestError，核心原因是 DeepSeek API 目前不支持 ProviderStrategy 所使用的 json_schema 结构化输出格式。

**问题根源：ProviderStrategy 与 DeepSeek 的不兼容**

- 在你的代码中，ProviderStrategy(ContractInfo) 会指示 LangChain 尝试调用模型提供商的原生结构化输出功能（即 response_format: {"type": "json_schema", ...}）。

- 然而，DeepSeek 的 API 明确不支持 json_schema 这种响应格式，当收到此类请求时会直接拒绝并返回 400 错误。错误信息 This response_format type is unavailable now 正源于此。

详细解释：当传入 ProviderStrategy(ContractInfo) 时，**LangChain 会先将 ContractInfo 这个 Pydantic 模型内部转换为一个 JSON Schema 对象**，然后以 response_format: {"type": "json_schema", ...} 的形式发送给 DeepSeek API。DeepSeek 拒绝的是这个 API 层面的 json_schema 请求类型。

### 3.2 ToolStrategy策略：

In [20]:
from pydantic import BaseModel, Field
from langchain.agents import create_agent
from langchain.agents.structured_output import ProviderStrategy, ToolStrategy
from langchain.messages import HumanMessage
from rich import print as rprint
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
import os

load_dotenv(override=True)

# 1.模型初始化
# model = init_chat_model(
#     model="gpt-5.4-mini",
#     model_provider="openai",
#     api_key=os.getenv("CLOSEAI_API_KEY"),
#     base_url=os.getenv("CLOSEAI_BASE_URL")
# )

model = init_chat_model(
    model="deepseek-v4-flash",
    model_provider="deepseek",
    # reasoning_effort="none",  # 关闭思考模式
    extra_body={"thinking": {"type": "disabled"}},  # 第二种写法关闭思考模式
    api_key=os.getenv("DEEPSEEK_API_KEY"),
    base_url=os.getenv("DEEPSEEK_BASE_URL")
)

# 2.使用 Pydantic 结构化方式定义
class ContractInfo(BaseModel):
    """用户的联系方式"""
    name : str = Field(description="用户的姓名")
    email : str = Field(description="用户的邮箱")
    phone : str = Field(description="用户的电话")

agent = create_agent(
    model = model,
    response_format=ToolStrategy(ContractInfo)
)


response = agent.invoke({
    "messages": [
        # {"role":"user","content":"从以下信息中提取用户信息，小明的邮箱是shkstart@atguigu.com,电话是13012341234"}
        HumanMessage(content="从以下信息中提取用户信息，小明的邮箱是shkstart@atguigu.com,电话是13012341234")
    ]
})

rprint(response)
print(type(response["structured_response"]))
print(response["structured_response"])

{
    'messages': [
        HumanMessage(
            content='从以下信息中提取用户信息，小明的邮箱是shkstart@atguigu.com,电话是13012341234',
            additional_kwargs={},
            response_metadata={},
            id='503cdc74-417b-49d0-9cda-c6dafad657c5'
        ),
        AIMessage(
            content='',
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 74,
                    'prompt_tokens': 337,
                    'total_tokens': 411,
                    'completion_tokens_details': None,
                    'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 128},
                    'prompt_cache_hit_tokens': 128,
                    'prompt_cache_miss_tokens': 209
                },
                'model_provider': 'deepseek',
                'model_name': 'deepseek-flash',
                'system_fingerprint': 'aeb56401ca74e127821c4f9126dcb669',
                'id': '015b7534-d81c-4e57-91e5-202b4bfa31ba',
                'finish_reason': 'tool_calls',
                'logprobs': None
            },
            id='lc_run--01a12df8-e5d9-7ed3-978c-0ac9cbee57c8-0',
            tool_calls=[
                {
                    'name': 'ContractInfo',
                    'args': {'name': '小明', 'email': 'shkstart@atguigu.com', 'phone': '13012341234'},
                    'id': 'call_00_BzWkgDXu5X0GhFGVObNZ3651',
                    'type': 'tool_call'
                }
            ],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 337,
                'output_tokens': 74,
                'total_tokens': 411,
                'input_token_details': {'cache_read': 128},
                'output_token_details': {}
            }
        ),
        ToolMessage(
            content="Returning structured response: name='小明' email='shkstart@atguigu.com' phone='13012341234'",
            name='ContractInfo',
            id='782ea28d-eb9a-4309-b8e4-0be03b328775',
            tool_call_id='call_00_BzWkgDXu5X0GhFGVObNZ3651'
        )
    ],
    'structured_response': ContractInfo(name='小明', email='shkstart@atguigu.com', phone='13012341234')
}

<class '__main__.ContractInfo'>
name='小明' email='shkstart@atguigu.com' phone='13012341234'


### 3.3 AutoStrategy / Type策略：

In [24]:
from pydantic import BaseModel, Field
from langchain.agents import create_agent
from langchain.agents.structured_output import ProviderStrategy, ToolStrategy, AutoStrategy
from langchain.messages import HumanMessage
from rich import print as rprint
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
import os

load_dotenv(override=True)

# 1.模型初始化
# model = init_chat_model(
#     model="gpt-5.4-mini",
#     model_provider="openai",
#     api_key=os.getenv("CLOSEAI_API_KEY"),
#     base_url=os.getenv("CLOSEAI_BASE_URL")
# )

model = init_chat_model(
    model="deepseek-v4-flash",
    model_provider="deepseek",
    # reasoning_effort="none",  # 关闭思考模式
    extra_body={"thinking": {"type": "disabled"}},  # 第二种写法关闭思考模式
    api_key=os.getenv("DEEPSEEK_API_KEY"),
    base_url=os.getenv("DEEPSEEK_BASE_URL")
)

# 2.使用 Pydantic 结构化方式定义
class ContractInfo(BaseModel):
    """用户的联系方式"""
    name : str = Field(description="用户的姓名")
    email : str = Field(description="用户的邮箱")
    phone : str = Field(description="用户的电话")

agent = create_agent(
    model = model,
    response_format=AutoStrategy(ContractInfo)
    # response_format=ContractInfo  # 不推荐大家使用
)


response = agent.invoke({
    "messages": [
        # {"role":"user","content":"从以下信息中提取用户信息，小明的邮箱是shkstart@atguigu.com,电话是13012341234"}
        HumanMessage(content="从以下信息中提取用户信息，小明的邮箱是shkstart@atguigu.com,电话是13012341234")
    ]
})

rprint(response)
print(type(response["structured_response"]))
print(response["structured_response"])

{
    'messages': [
        HumanMessage(
            content='从以下信息中提取用户信息，小明的邮箱是shkstart@atguigu.com,电话是13012341234',
            additional_kwargs={},
            response_metadata={},
            id='8da0e68d-1969-4910-85ce-5c76e4df220c'
        ),
        AIMessage(
            content='',
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 74,
                    'prompt_tokens': 337,
                    'total_tokens': 411,
                    'completion_tokens_details': None,
                    'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 128},
                    'prompt_cache_hit_tokens': 128,
                    'prompt_cache_miss_tokens': 209
                },
                'model_provider': 'deepseek',
                'model_name': 'deepseek-flash',
                'system_fingerprint': 'aeb56401ca74e127821c4f9126dcb669',
                'id': '527c00f7-2c40-4a1b-955c-41f0f0ca7c37',
                'finish_reason': 'tool_calls',
                'logprobs': None
            },
            id='lc_run--01a12e05-03fc-71b2-8407-a181dd5b98b6-0',
            tool_calls=[
                {
                    'name': 'ContractInfo',
                    'args': {'name': '小明', 'email': 'shkstart@atguigu.com', 'phone': '13012341234'},
                    'id': 'call_00_8qaUSjX9T3xnWO4bezla8395',
                    'type': 'tool_call'
                }
            ],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 337,
                'output_tokens': 74,
                'total_tokens': 411,
                'input_token_details': {'cache_read': 128},
                'output_token_details': {}
            }
        ),
        ToolMessage(
            content="Returning structured response: name='小明' email='shkstart@atguigu.com' phone='13012341234'",
            name='ContractInfo',
            id='8db6efe8-ac55-4b0a-a367-86a8bd03c702',
            tool_call_id='call_00_8qaUSjX9T3xnWO4bezla8395'
        )
    ],
    'structured_response': ContractInfo(name='小明', email='shkstart@atguigu.com', phone='13012341234')
}

<class '__main__.ContractInfo'>
name='小明' email='shkstart@atguigu.com' phone='13012341234'
